# Ratings SoFIFA des équipes du top 100 UEFA

Objectif : une table **une ligne par équipe** avec ses notes par catégorie
(`overallRating`, `attackRating`, `midfieldRating`, `defenseRating`) plus des extras utiles
(prestige, valeur du club, âge moyen du onze), pour les 100 clubs de `uefa_top100_2027.csv`.

**Méthode en 4 étapes**
1. **Annuaire** — scraper la liste complète des équipes SoFIFA (`id`, `nom`, `ligue`) via seleniumbase.
2. **Matching** — rapprocher les 100 noms du CSV des noms SoFIFA (normalisation + fuzzy), avec revue manuelle.
3. **Fetch** — appeler `api.sofifa.net/team/{id}` pour chaque équipe et parser les champs de notes.
4. **Stockage** — CSV + table Neon `team_sofifa_ratings`.

**Garde-fous**
- `api.sofifa.net` est derrière Cloudflare → on réutilise ton pattern seleniumbase (warmup + `fetch` dans le
  contexte JS de la page, qui porte le cookie `cf_clearance`). Une **fenêtre Brave visible** s'ouvre ; clique la
  case Turnstile si elle apparaît.
- Rythme volontairement lent (`sleep` entre chaque requête) — 100 équipes, pas de matraquage.
- L'annuaire est **mis en cache** (`sofifa_team_directory.csv`) : on ne le re-scrape pas si le fichier existe.

In [ ]:
from pathlib import Path
import os, re, json, time
from difflib import SequenceMatcher, get_close_matches

import pandas as pd
from bs4 import BeautifulSoup
from unidecode import unidecode
from seleniumbase import Driver
from sqlalchemy import create_engine

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

# Chemin du binaire Brave (repris de players_stats.ipynb).
BROWSER = "/Applications/Brave Browser.app/Contents/MacOS/Brave Browser"

PROJECT_ROOT = Path.cwd()

# Le CSV top100 vit dans Futbol/src/ ; on le cherche à plusieurs emplacements probables.
TOP100_CSV = next(
    (c for c in [
        PROJECT_ROOT / "src" / "uefa_top100_2027.csv",
        PROJECT_ROOT.parent.parent / "src" / "uefa_top100_2027.csv",
        PROJECT_ROOT / "uefa_top100_2027.csv",
        *PROJECT_ROOT.parent.parent.rglob("uefa_top100_2027.csv"),
    ] if c.exists()),
    None,
)
if TOP100_CSV is None:
    raise FileNotFoundError("uefa_top100_2027.csv introuvable — ajuste le chemin.")

DIRECTORY_CSV = PROJECT_ROOT / "sofifa_team_directory.csv"
RATINGS_CSV = PROJECT_ROOT / "team_sofifa_ratings.csv"
print("top100 :", TOP100_CSV)
print("annuaire (cache) :", DIRECTORY_CSV)
print("sortie ratings :", RATINGS_CSV)

In [ ]:
top100 = pd.read_csv(TOP100_CSV)
print(top100.shape)
display(top100.head())

## 1. Annuaire SoFIFA (id → nom → ligue)

On pagine `sofifa.com/teams` dans la fenêtre Brave et on récupère tous les liens `/team/{id}/…`.

⚠️ **Point fragile** : la structure HTML de SoFIFA change parfois. Si l'annuaire revient avec très peu de lignes,
c'est que le paramètre `offset` ou le sélecteur a changé — ajuste `per_page` / le `select()`. On parse volontairement
par **regex sur les href** (robuste) plutôt que par classes CSS.

In [ ]:
def norm(s: str) -> str:
    # minuscule, sans accents, sans ponctuation, espaces simples
    s = unidecode(str(s)).lower()
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def scrape_sofifa_directory(max_pages=40, per_page=60, sleep=1.4):
    rows = {}
    driver = Driver(uc=True, headless=False, binary_location=BROWSER)
    try:
        for p in range(max_pages):
            url = f"https://sofifa.com/teams?offset={p * per_page}"
            driver.uc_open_with_reconnect(url, reconnect_time=6)   # passe Cloudflare
            try:
                driver.uc_gui_click_captcha()
            except Exception:
                pass
            driver.sleep(sleep)
            soup = BeautifulSoup(driver.get_page_source(), "lxml")

            found = 0
            for a in soup.select('a[href^="/team/"]'):
                m = re.match(r"^/team/(\d+)/", a.get("href", ""))
                name = a.get_text(strip=True)
                if not m or not name:
                    continue
                tid = int(m.group(1))
                if tid in rows:
                    continue
                league = ""
                tr = a.find_parent("tr")
                if tr:
                    la = tr.select_one('a[href^="/league/"]')
                    if la:
                        league = la.get_text(strip=True)
                rows[tid] = {"sofifa_id": tid, "sofifa_name": name, "league": league}
                found += 1

            print(f"page {p:>2}  offset {p * per_page:>4}  +{found:>3}  (total {len(rows)})")
            if found == 0:      # plus de nouvelles équipes -> fin
                break
    finally:
        driver.quit()
    return pd.DataFrame(rows.values())


if DIRECTORY_CSV.exists():
    sofifa_teams = pd.read_csv(DIRECTORY_CSV)
    print("annuaire chargé du cache :", sofifa_teams.shape)
else:
    sofifa_teams = scrape_sofifa_directory()
    sofifa_teams.to_csv(DIRECTORY_CSV, index=False)
    print("annuaire scrapé et mis en cache :", sofifa_teams.shape)

display(sofifa_teams.head())

## 2. Matching top100 → id SoFIFA

Normalisation + quelques alias connus, puis fuzzy (`difflib`) pour le reste. Les lignes à faible score sont
**marquées `needs_review`** : vérifie-les et corrige via `OVERRIDES` avant de lancer le fetch.

In [ ]:
# Alias : nom du CSV (normalisé) -> nom SoFIFA (normalisé). À compléter selon les cas.
ALIASES = {
    "paris": "paris saint germain",
    "internazionale": "inter",
    "inter milan": "inter",
    "man city": "manchester city",
    "man united": "manchester united",
    "man utd": "manchester united",
    "spurs": "tottenham hotspur",
    "atletico madrid": "atletico de madrid",
    "sporting": "sporting cp",
    "wolves": "wolverhampton wanderers",
}

# Corrections manuelles après revue : nom exact du CSV (colonne 'club') -> sofifa_id
OVERRIDES = {}

sofifa_teams = sofifa_teams.copy()
sofifa_teams["key"] = sofifa_teams["sofifa_name"].map(lambda n: ALIASES.get(norm(n), norm(n)))
key_to_id = dict(zip(sofifa_teams["key"], sofifa_teams["sofifa_id"]))
id_to_name = dict(zip(sofifa_teams["sofifa_id"], sofifa_teams["sofifa_name"]))
all_keys = sofifa_teams["key"].tolist()


def match_club(club: str):
    if club in OVERRIDES:
        sid = OVERRIDES[club]
        return sid, id_to_name.get(sid, ""), 1.0
    key = ALIASES.get(norm(club), norm(club))
    if key in key_to_id:
        return key_to_id[key], id_to_name[key_to_id[key]], 1.0
    cand = get_close_matches(key, all_keys, n=1, cutoff=0.0)
    if not cand:
        return None, None, 0.0
    best = cand[0]
    score = round(SequenceMatcher(None, key, best).ratio(), 2)
    sid = key_to_id[best]
    return sid, id_to_name.get(sid, ""), score


matched = pd.DataFrame([
    {"rank": r["rank"], "club": r["club"], "country": r["country"],
     **dict(zip(["sofifa_id", "sofifa_name", "score"], match_club(r["club"])))}
    for _, r in top100.iterrows()
])
matched["needs_review"] = matched["sofifa_id"].isna() | (matched["score"] < 0.72)

print(f"à vérifier : {int(matched['needs_review'].sum())} / {len(matched)}")
display(matched.sort_values(["needs_review", "score"], ascending=[False, True]))

**Boucle de correction** : pour chaque ligne marquée, retrouve le bon id (cherche le nom dans `sofifa_teams`),
remplis `OVERRIDES = {"Nom exact du CSV": 241, ...}` dans la cellule ci-dessus, puis **ré-exécute-la**. Recommence
jusqu'à `needs_review = 0`. Exemple pour trouver un id :

```python
sofifa_teams[sofifa_teams["sofifa_name"].str.contains("Barcelona", case=False, na=False)]
```

## 3. Fetch des ratings par équipe (`api.sofifa.net/team/{id}`)

On **navigue directement** le navigateur sur `https://api.sofifa.net/team/{id}` : la page affiche le JSON, on lit le
corps et on parse. Pas de fetch cross-origin, pas de paramètre `roster` à trouver. Le 1er appel dédouane Cloudflare
(clique le captcha) ; ensuite le cookie `cf_clearance` persiste dans la session, les appels suivants renvoient le
JSON directement (re-dédouanement automatique si un challenge repasse).

On garde les champs **niveau équipe** (les 4 notes + prestige, valeur, âges…) et on jette le tableau `players`.

> **Variante « historique »** : la page SoFIFA appelle en interne l'endpoint *same-origin*
> `sofifa.com/api/team/history?id={id}&roster={roster}` — c'est la **série temporelle** des notes (la tendance),
> pas le snapshot, et il faut d'abord lire le `roster` courant dans le HTML de la page équipe. À utiliser seulement
> si tu veux l'évolution des notes dans le temps ; pour une table courante, le snapshot ci-dessous suffit.

In [ ]:
# Champs niveau équipe à conserver (d'après la réponse d'exemple de l'API).
TEAM_FIELDS = [
    "id", "name", "country", "leagueId", "countryId", "playerCount",
    "transferBudget", "intPrestige", "domPrestige", "popularity", "youthDevelopment",
    "clubWorth", "xiAverageAge", "totalAverageAge",
    "overallRating", "attackRating", "midfieldRating", "defenseRating",
    "version", "roster",
]

def _body_text(driver):
    # une URL JSON s'affiche telle quelle dans le <body> ; on lit le texte brut.
    return driver.execute_script("return document.body ? document.body.innerText : ''") or ""


def _dedouane(driver, url):
    driver.uc_open_with_reconnect(url, reconnect_time=6)   # résout le challenge Cloudflare
    try:
        driver.uc_gui_click_captcha()
    except Exception:
        pass
    driver.sleep(2)


def fetch_team(driver, sofifa_id, warm=False):
    url = f"https://api.sofifa.net/team/{sofifa_id}"
    if warm:
        _dedouane(driver, url)
    else:
        driver.get(url)            # le cookie cf_clearance de la session suffit
        driver.sleep(0.7)
    raw = _body_text(driver)
    if not raw.lstrip().startswith("{"):   # un challenge est repassé -> on re-dédouane
        _dedouane(driver, url)
        raw = _body_text(driver)
    data = json.loads(raw)["data"]
    return {k: data.get(k) for k in TEAM_FIELDS}


def collect_ratings(matched_df, sleep=1.5):
    todo = matched_df.dropna(subset=["sofifa_id"]).copy()
    todo["sofifa_id"] = todo["sofifa_id"].astype(int)

    out = []
    driver = Driver(uc=True, headless=False, binary_location=BROWSER)
    try:
        warm = True   # 1er appel = warmup Cloudflare (clique le captcha si besoin)
        for _, r in todo.iterrows():
            sid = int(r["sofifa_id"])
            try:
                rec = fetch_team(driver, sid, warm=warm)
                warm = False
                rec.update({"rank": r["rank"], "csv_club": r["club"], "csv_country": r["country"]})
                out.append(rec)
                print(f"OK  {sid:>6}  {str(rec.get('name'))[:26]:<26}  "
                      f"OVR={rec.get('overallRating')}  ATT={rec.get('attackRating')}  "
                      f"MID={rec.get('midfieldRating')}  DEF={rec.get('defenseRating')}")
            except Exception as e:
                print(f"ERR {sid:>6}  {r['club']}: {type(e).__name__} {e}")
                warm = True   # forcer un re-dédouanement au prochain
            driver.sleep(sleep)
    finally:
        driver.quit()
    return pd.DataFrame(out)


team_ratings = collect_ratings(matched)
team_ratings.to_csv(RATINGS_CSV, index=False)
print("\nrécupéré :", team_ratings.shape, "->", RATINGS_CSV)
display(team_ratings.head(20))

## 4. Stockage dans Neon (optionnel)

Écrit `team_sofifa_ratings` dans la base. On lit l'URL de connexion du `.env` local (jamais affichée).

In [ ]:
def load_env_file(path):
    values = {}
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            values[k.strip()] = v.strip().strip('"').strip("'")
    return values


env = load_env_file(PROJECT_ROOT / ".env")
url = (env.get("NEON_DATABASE_URL") or env.get("DATABASE_URL") or env.get("POSTGRES_URL") or "")
if not url:
    raise RuntimeError("Pas d'URL Neon dans .env")
url = url.replace("postgresql://", "postgresql+psycopg2://", 1).replace("postgres://", "postgresql+psycopg2://", 1)

engine = create_engine(url, pool_pre_ping=True)
team_ratings.to_sql("team_sofifa_ratings", engine, schema="public", if_exists="replace", index=False)
print("écrit dans public.team_sofifa_ratings :", team_ratings.shape)

## Notes

- **Fenêtre Brave visible obligatoire** (`headless=False`) : c'est ce qui passe Cloudflare. Ne la ferme pas
  pendant l'exécution.
- **Vérifie le matching** avant le fetch — un mauvais id renvoie les notes d'un autre club sans erreur.
- La colonne `players` de l'API est **volontairement ignorée** ici (tu as déjà les joueurs en base) ; si un jour tu
  la veux, ajoute-la dans `fetch_team`.
- Sélecteurs SoFIFA fragiles : si l'annuaire ou le fetch se vide, l'HTML/endpoint a bougé — c'est le seul endroit à
  retoucher.